# Deep Learning Assignment 1

**Import `torch` and DS helpers**:


In [18]:
from typing import Protocol
import torch
from dataclasses import dataclass
from IPython.display import Math, display

Tensor = torch.Tensor  # avoid typing `torch.Tensor` at every function decleration

## 1. Data structures for `Layer`, `Activation` & `Loss`

We will create an MLP class which holds information about the network, methods for forward and backward pass, and a cache for repitative computation.

The MLP needs to work with any number of layers, each can take any size and any activation function. Additionaly it needs to work with 3 different loss functions. To accomplish that, uppon initilization, it will require the following information:

- A `[Layer]` array - where each `Layer` object has a `size` and an `activation`
- The input's shape
- The loss function.

It will use the layer's sizes to work out the Weight matrices shapes, and initilize then randomaly from a normal distribution with a SEED.

We will use type-safe data structures to make life easier when debugging later.


### 1.1 Activation functions

We will use a `Protocol` for a general `Activation` with `.forward()` and `.derivative()`, then define `Relu`, `Identity` and `Sigmoid`


In [19]:
class Activation(Protocol):  # Later the `MLP` class could use a `list(Activation)` type
    def forward(self, x: Tensor) -> Tensor: ...
    def derivative(self, x: Tensor) -> Tensor: ...
    def __repr__(self) -> str: ...


class ReLU(Activation):
    def forward(self, x: Tensor) -> Tensor:
        return x.clamp(min=0)

    def derivative(self, x: Tensor) -> Tensor:
        return (x > 0).to(x.dtype)

    def __repr__(self) -> str:
        return "ReLU"


class Identity(Activation):
    def forward(self, x: Tensor) -> Tensor:
        return x

    def derivative(self, x: Tensor) -> Tensor:
        return torch.ones_like(x)

    def __repr__(self) -> str:
        return "I"


class Sigmoid(Activation):
    def forward(self, x: Tensor) -> Tensor:
        return 1 / (1 + torch.exp(-x))

    def derivative(self, x: Tensor) -> Tensor:
        s = self.forward(x)
        return s * (1 - s)

    def __repr__(self) -> str:
        return "Sigmoid"

### 1.2 Loss function

We will use a `Loss` protocol similar to `Activation` but with an extra `target` input for the functions. The Losses will also inharit a method to verify that the shapes of `y_hat` and `y` match -- and raise a helpfull `ValueError` when they dont.


In [20]:
class Loss(Protocol):
    def verify_dim_match(self, y_hat: Tensor, y: Tensor) -> None:
        if y_hat.shape != y.shape:
            raise ValueError(
                f"Expected matching shapes, got {y_hat.shape} and {y.shape}. Go check your dimensions ;)"
            )

    def forward(self, y_hat: Tensor, y: Tensor) -> Tensor: ...
    def derivative(self, y_hat: Tensor, y: Tensor) -> Tensor: ...


class RSS(Loss):
    def forward(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return ((y_hat - y) ** 2).sum()

    def derivative(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return 2 * (y_hat - y)


class LogLoss(Loss):  # assumes y is a binary and y_hat in range(0,1)
    def forward(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return -1 * (y * torch.log(y_hat) + (1 - y) * torch.log(1 - y_hat))

    def derivative(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return -1 * (y / y_hat) + (1 - y) / (1 - y_hat)


class AbsoluteError(Loss):
    def forward(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return torch.abs(y_hat - y)

    def derivative(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return (y_hat - y).sign()

### 1.3 Layer


In [21]:
@dataclass
class Layer:
    size: int
    activation: Activation

## 2. Designing the MLP

### **Assumptions**

The desired network is fully conncected, where each layer has an activation function. The final layer is the output. The network assumes a single column vector as input each pass, or potentialy a batch matrix in which case the weights will be tranposed. So a weight $W \in \mathbb{R}^{n \times m}$ is a linear map from an $m$-sized layer to an $n$ sized one.


### 2.1 Initing Weights and display


In [ ]:
from pydantic import InstanceOf


class Network:
    def __init__(self, in_size, loss: Loss, layers: list[Layer]):
        self.in_size = in_size
        self.layers = layers
        self.L = loss
        self.layers_num = len(layers)
        self.init_weights()

    def init_weights(self):
        g = torch.Generator().manual_seed(42)
        in_size = self.in_size
        weights: list[Tensor] = []
        biases: list[Tensor] = []
        for i in range(self.layers_num):
            out_size = self.layers[i].size
            W = torch.randn(out_size, in_size, generator=g)
            b = torch.zeros(out_size)
            weights.append(W)
            biases.append(b)
            in_size = out_size

        self.y_size = out_size
        self.weights = weights
        self.biases = biases

    def display(self):
        l = self.layers_num - 1

        def tex(t, p=2):
            t = t.reshape(-1, 1) if t.dim() == 1 else t
            body = r" \\ ".join(
                " & ".join(f"{round(v, p):g}" for v in row) for row in t.tolist()
            )
            return rf"\begin{{bmatrix}} {body} \end{{bmatrix}}"

        def build_tex(l):
            if l < 0:
                return r"\mathbf{x}"
            act = repr(self.layers[l].activation)
            W, b = self.weights[l], self.biases[l]

            inner_tex = rf"{tex(W)} {build_tex(l - 1)} + {tex(b)}"

            if l == self.layers_num - 1 and isinstance(
                self.layers[l].activation, Identity
            ):
                return inner_tex

            return rf"\operatorname{{{act}}}\left( {tex(W)} {build_tex(l - 1)} + {tex(b)} \right)"

        display(Math(r"\mathbf{y} = " + build_tex(self.layers_num - 1)))


In [25]:
loss = RSS()
relu = ReLU()
identity = Identity()
layers = [Layer(3, relu), Layer(3, relu), Layer(2, identity)]
in_size = 2

net = Network(in_size, loss, layers)
print(net)

net.display()

<IPython.core.display.Math object>